In [60]:
# Import required libraries
import requests        # To fetch data from USGS API
import pandas as pd    # To handle tabular data
import plotly.graph_objects as go # For interactive plotting
import json # Helps display API response in readable format


In [61]:
# -------------------------------
# API Request Parameters
# -------------------------------
site_no = "USGS-08330000"    # USGS site ID (Rio Grande at Albuquerque)
parameter_cd = "00060"        # Parameter code for discharge
statistic_id = "00003"        # Statistic ID for daily mean
start_date = "2025-05-01"    # Start date
end_date = "2025-09-30"      # End date
limit = 5000                  # High enough to fetch all daily values in one request

# Base URL for the OGC API daily endpoint
base_url = "https://api.waterdata.usgs.gov/ogcapi/v0/collections/daily/items"


In [62]:
# -------------------------------
# Fetch daily data from OGC API in a single request
# -------------------------------
params = {
    "monitoring_location_id": site_no,
    "parameter_code": parameter_cd,
    "statistic_id": statistic_id,
    "time": f"{start_date}/{end_date}",
    "limit": str(limit)
    # "api_key": "YOUR_KEY"  # Optional: use for higher request limits
}

# Make GET request to the API
response = requests.get(base_url, params=params)


# Raise an exception if request fails
if response.status_code != 200:
    raise Exception(f"Request failed with status {response.status_code}: {response.text}")

# Parse JSON response
data = response.json()


In [63]:
# -------------------------------
# FULL RESPONSE INSPECTION CELL
# -------------------------------

print("\n========== TOP LEVEL KEYS ==========")
print(list(data.keys()))


# -------------------------------
# LINKS
# -------------------------------
print("\n========== LINKS ==========")
links = data.get("links", [])
if links:
    for i, link in enumerate(links, 1):
        print(f"{i}. {link}")
else:
    print("No links found")


# -------------------------------
# FEATURES INFO
# -------------------------------
features = data.get("features", [])
print("\n========== FEATURES ==========")
print("Number of features returned:", len(features))


# -------------------------------
# FEATURE STRUCTURE
# -------------------------------
if features:
    print("\n========== FIRST FEATURE KEYS ==========")
    print(list(features[0].keys()))

    properties = features[0].get("properties", {})

    print("\n========== PROPERTIES KEYS ==========")
    print(list(properties.keys()))

    print("\n========== SAMPLE OBSERVATION ==========")
    print("Timestamp :", properties.get("time"))
    print("Value     :", properties.get("value"))
    print("Parameter :", properties.get("parameter_code"))
    print("Statistic :", properties.get("statistic_id"))
else:
    print("No features found in response")


# -------------------------------
# PRETTY JSON PREVIEW
# -------------------------------
print("\n========== PRETTY JSON PREVIEW (first feature) ==========")
if features:
    print(json.dumps(features[0], indent=2))
else:
    print("Nothing to preview")




========== TOP LEVEL KEYS ==========
['type', 'features', 'numberReturned', 'links', 'timeStamp']

========== LINKS ==========
1. {'type': 'application/geo+json', 'rel': 'self', 'title': 'This document as GeoJSON', 'href': 'https://api.waterdata.usgs.gov/ogcapi/v0/collections/daily/items?f=json&monitoring_location_id=USGS-08330000&parameter_code=00060&statistic_id=00003&time=2025-05-01%2F2025-09-30&limit=5000'}
2. {'rel': 'alternate', 'type': 'application/ld+json', 'title': 'This document as RDF (JSON-LD)', 'href': 'https://api.waterdata.usgs.gov/ogcapi/v0/collections/daily/items?f=jsonld&monitoring_location_id=USGS-08330000&parameter_code=00060&statistic_id=00003&time=2025-05-01%2F2025-09-30&limit=5000'}
3. {'type': 'text/html', 'rel': 'alternate', 'title': 'This document as HTML', 'href': 'https://api.waterdata.usgs.gov/ogcapi/v0/collections/daily/items?f=html&monitoring_location_id=USGS-08330000&parameter_code=00060&statistic_id=00003&time=2025-05-01%2F2025-09-30&limit=5000'}
4. {'

In [64]:
# -------------------------------
# Convert API response to Pandas DataFrame
# -------------------------------

# Extract 'features' from GeoJSON response
records = []
for f in data.get("features", []):
    p = f["properties"]
    records.append({
        "date": p.get("time"),                # Date of observation
        "value": p.get("value"),              # Discharge value (cfs)
        "parameter_code": p.get("parameter_code"),
        "statistic_id": p.get("statistic_id")
    })
    
# Create DataFrame
df = pd.DataFrame(records)

# Convert 'date' column to datetime type
df["date"] = pd.to_datetime(df["date"])

# Sort data by date
df = df.sort_values("date").reset_index(drop=True)

# Add a 'month' column for monthly aggregation
df['month'] = df['date'].dt.month


In [65]:
# -------------------------------
# Save the DataFrame to a CSV file
# -------------------------------
year_label = pd.to_datetime(start_date).year        # Get the year from the start date
csv_filename = f"{site_no}_{year_label}_daily.csv"  # Create filename using site number and year
df.to_csv(csv_filename, index=False)                # Save DataFrame to CSV without row index
print(f"Data saved to {csv_filename}")              # Print confirmation message

Data saved to USGS-08330000_2025_daily.csv


In [66]:
# --------------------------------------------------------------
# clean values, compute monthly aggregates,
# and report overall min/max plus total monthly extremes.
# --------------------------------------------------------------

# Convert discharge values to numeric (float), invalid entries become NaN
df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Drop rows where value is NaN
df = df.dropna(subset=['value'])

# Reset month column just in case
df['month'] = df['date'].dt.month

# Compute monthly stats and include parameter_code and statistic_id
monthly_stats = df.groupby('month').agg(
    parameter_code=('parameter_code', 'first'),
    statistic_id=('statistic_id', 'first'),
    min=('value', 'min'),
    max=('value', 'max'),
    mean=('value', 'mean'),
    total_discharge=('value', 'sum'),
).reset_index()

# Round ONLY the mean column to 2 decimals
monthly_stats['mean'] = monthly_stats['mean'].round(2)

# Add month name
monthly_stats['Month'] = pd.to_datetime(monthly_stats['month'], format='%m').dt.strftime('%b')

# Reorder columns (Month name, parameter_code, statistic_id, stats...)
monthly_stats = monthly_stats[[
    'Month', 'parameter_code', 'statistic_id', 'min', 'max', 'mean', 'total_discharge'
 ]]

display(monthly_stats)

# Compute overall min/max
overall_min = df['value'].min()
overall_max = df['value'].max()

# Find all months where min and max occur
min_months_list = df[df['value'] == overall_min]['date'].dt.strftime('%b').unique()
max_months_list = df[df['value'] == overall_max]['date'].dt.strftime('%b').unique()

# Join with commas if multiple months
min_months_str = ', '.join(min_months_list)
max_months_str = ', '.join(max_months_list)

print(
    f"Overall minimum discharge in {year_label} ({min_months_str}): "
    f"{overall_min:.2f} cfs"
 )

print(
    f"Overall maximum discharge in {year_label} ({max_months_str}): "
    f"{overall_max:.2f} cfs"
 )

# Min / Max TOTAL monthly discharge
min_total_row = monthly_stats.loc[monthly_stats['total_discharge'].idxmin()]
max_total_row = monthly_stats.loc[monthly_stats['total_discharge'].idxmax()]

print(
    f"Minimum total monthly discharge ({min_total_row['Month']}): "
    f"{min_total_row['total_discharge']:.2f} cfs"
 )

print(
    f"Maximum total monthly discharge ({max_total_row['Month']}): "
    f"{max_total_row['total_discharge']:.2f} cfs"
 )


,Month,parameter_code,statistic_id,min,max,mean,total_discharge
0,May,00060,00003,333.00,1010.0,558.52,17314.00
1,Jun,00060,00003,151.00,820.0,411.63,12349.00
2,Jul,00060,00003,0.00,565.0,51.83,1606.86
3,Aug,00060,00003,0.00,259.0,32.43,1005.25
4,Sep,00060,00003,0.31,365.0,103.15,3094.64


Overall minimum discharge in 2025 (Jul, Aug): 0.00 cfs
Overall maximum discharge in 2025 (May): 1010.00 cfs
Minimum total monthly discharge (Aug): 1005.25 cfs
Maximum total monthly discharge (May): 17314.00 cfs


In [67]:
# --------------------------------------------------------------
# Interactive Plot: Daily Water Discharge
# --------------------------------------------------------------

date_col = "date"  # Column name for dates
value_column = "value"  # Column name for discharge values

# Find min and max values and their months
min_value = df['value'].min() 
max_value = df['value'].max() 
min_months = df[df['value'] == min_value]['date'].dt.strftime('%b').unique()  # Months with min value
max_months = df[df['value'] == max_value]['date'].dt.strftime('%b').unique()  # Months with max value

min_month_str = ', '.join(min_months)  # String of min months
max_month_str = ', '.join(max_months)  # String of max months

import plotly.graph_objects as go
fig = go.Figure()

# Main daily discharge line
fig.add_trace(
    go.Scatter(
        x=df[date_col],  
        y=df[value_column],
        mode="lines+markers",
        name="Daily Discharge",
        marker=dict(size=4), 
        line=dict(color="blue"), 
        hovertemplate="<b>Date:</b> %{x|%Y-%m-%d}<br><b>Discharge:</b> %{y:.2f} cfs<extra></extra>"  # Tooltip format
    )
)

# Annotation above plot area
fig.add_annotation(
    text=f"For Period: {start_date} to {end_date}<br>Min Discharge: {min_value:.2f} cfs ({min_month_str})<br>Max Discharge: {max_value:.2f} cfs ({max_month_str})",  # Info box
    xref="paper",
    yref="paper",
    x=1.0,  
    y=1.0,
    showarrow=False,
    xanchor="right",
    yanchor="bottom",
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="gray",
    borderwidth=1
)

# Layout settings
fig.update_layout(
    title=f"Interactive Daily Discharge - Site {site_no} ({year_label})",  # Plot title
    xaxis_title="Date",  # X-axis label
    yaxis_title="Discharge (cfs)",  # Y-axis label
    hovermode="x unified",  # Unified hover
    template="plotly_white",  # Plot style
    xaxis=dict(
        dtick="M1",  # Tick every month
        tickformat="%b %Y",  # Month and year format
        tickangle=-45  # Tilt labels
    ),
    height=600,
    margin=dict(t=120)
)

fig.show()  # Display the plot